# Sprint 1 Validation Notebook

Visual verification of all Sprint 1 exit criteria:
1. **3,520 samples parsed** without error
2. **Features extracted**: B, tau_fast, tau_slow, centroid, MFCC, room IR, excitation
3. **Resynthesis blocking test**: MR-STFT distance ≤ 1.5× median RR distance
4. **Train/val/test split** fixed in manifest
5. **≥5,000 MIDI-audio pairs** rendered via sfizz CLI
6. **Output files** generated (features.h5, midi_pairs.h5, manifest.parquet)

In [ ]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))
from piano_neuronal.config import FEATURES_H5_PATH, MANIFEST_PATH, MIDI_PAIRS_H5_PATH, OUTPUT_DIR

## 1. Files Parsed

In [ ]:
if MANIFEST_PATH.exists():
    df = pd.read_parquet(MANIFEST_PATH)
    expected = 88 * 5 * 2 * 2 * 2  # notes × velocities × RRs × mics × pedals
    print(f'Expected: {expected}, Found: {len(df)}')
    print(f'PASS' if len(df) == expected else 'FAIL')
    print(f'\nColumns: {list(df.columns)}')
    print(f'\nFirst 5 rows:')
    display(df.head())
else:
    print('FAIL: Manifest not found')

## 2. Feature Extraction Verification

In [ ]:
if FEATURES_H5_PATH.exists():
    with h5py.File(FEATURES_H5_PATH, 'r') as hf:
        groups = list(hf.keys())
        print(f'Total groups in HDF5: {len(groups)}')
        print(f'\nSample group: {groups[0]}')
        
        grp = hf[groups[0]]
        print(f'\nAttributes:')
        for k, v in grp.attrs.items():
            print(f'  {k}: {v}')
        
        print(f'\nDatasets:')
        for k in grp.keys():
            ds = grp[k]
            print(f'  {k}: shape={ds.shape}, dtype={ds.dtype}')
else:
    print('FAIL: Features HDF5 not found')

## 3a. Inharmonicity B vs Note

In [ ]:
if MANIFEST_PATH.exists():
    df = pd.read_parquet(MANIFEST_PATH)
    
    # Plot B vs MIDI note (should increase toward high register)
    fig, ax = plt.subplots(figsize=(12, 5))
    for vel in df['velocity_layer'].unique():
        mask = df['velocity_layer'] == vel
        subset = df[mask].sort_values('midi_note')
        ax.scatter(subset['midi_note'], subset['B'], alpha=0.3, s=10, label=vel)
    
    ax.set_xlabel('MIDI Note')
    ax.set_ylabel('Inharmonicity B')
    ax.set_title('Inharmonicity B vs Note (should increase toward high register)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3b. Decay: tau_fast and tau_slow vs Note

In [ ]:
if MANIFEST_PATH.exists():
    df = pd.read_parquet(MANIFEST_PATH)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # tau_fast (prompt soundboard decay)
    for vel in sorted(df['velocity_layer'].unique()):
        mask = df['velocity_layer'] == vel
        subset = df[mask].sort_values('midi_note')
        ax1.scatter(subset['midi_note'], subset['tau_fast'], alpha=0.3, s=10, label=vel)
    ax1.set_xlabel('MIDI Note')
    ax1.set_ylabel('τ_fast (s)')
    ax1.set_title('τ_fast (prompt soundboard decay)')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # tau_slow (aftersound)
    for vel in sorted(df['velocity_layer'].unique()):
        mask = df['velocity_layer'] == vel
        subset = df[mask].sort_values('midi_note')
        ax2.scatter(subset['midi_note'], subset['tau_slow'], alpha=0.3, s=10, label=vel)
    ax2.set_xlabel('MIDI Note')
    ax2.set_ylabel('τ_slow (s)')
    ax2.set_title('τ_slow (aftersound decay)')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 3c. Spectral Centroid vs Velocity

In [ ]:
if MANIFEST_PATH.exists():
    df = pd.read_parquet(MANIFEST_PATH)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    # Box plot of centroid by velocity layer
    vel_order = ['Pianissimo', 'Piano', 'MezzoPiano', 'MezzoForte', 'Forte']
    existing_vels = [v for v in vel_order if v in df['velocity_layer'].unique()]
    data_by_vel = [df[df['velocity_layer'] == v]['spectral_centroid_mean'].values for v in existing_vels]
    ax.boxplot(data_by_vel, labels=existing_vels)
    ax.set_ylabel('Spectral Centroid (Hz)')
    ax.set_title('Spectral Centroid vs Velocity (should increase with velocity)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3d. MFCC Distribution

In [ ]:
if FEATURES_H5_PATH.exists():
    with h5py.File(FEATURES_H5_PATH, 'r') as hf:
        # Collect MFCCs from a sample of notes
        mfcc_samples = []
        note_labels = []
        groups = list(hf.keys())
        
        for gname in groups[:50]:
            if f'feat_mfcc' in hf[gname]:
                mfcc_samples.append(hf[gname]['feat_mfcc'][:])
                note_labels.append(hf[gname].attrs.get('midi_note', 0))
        
        if mfcc_samples:
            print(f'Collected {len(mfcc_samples)} MFCC samples')
            print(f'MFCC shape example: {mfcc_samples[0].shape}')
            
            # Plot average MFCC across notes
            fig, ax = plt.subplots(figsize=(10, 5))
            for mfcc, note in zip(mfcc_samples, note_labels):
                ax.plot(mfcc, alpha=0.3, linewidth=0.5)
            ax.set_xlabel('MFCC Coefficient')
            ax.set_ylabel('Value')
            ax.set_title('MFCC profiles across notes')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print('No MFCC data found in HDF5')

## 4. Train/Val/Test Split Verification

In [ ]:
if MANIFEST_PATH.exists():
    df = pd.read_parquet(MANIFEST_PATH)
    
    # Split distribution
    print('Split distribution:')
    print(df['split'].value_counts())
    print()
    
    # Check MezzoPiano is entirely in test
    from piano_neuronal.config import SPLIT_VELOCITY_TEST
    mp_rows = df[df['velocity_layer'] == SPLIT_VELOCITY_TEST]
    mp_splits = mp_rows['split'].unique()
    mp_all_test = len(mp_splits) == 1 and mp_splits[0] == 'test'
    print(f"'{SPLIT_VELOCITY_TEST}' entirely in test: {mp_all_test}")
    
    # Check C notes (midi % 12 == 0) are in test
    c_notes = df[df['midi_note'] % 12 == 0]
    c_not_train = (c_notes['split'] != 'train').all()
    print(f'C notes (midi%12==0) not in train: {c_not_train}')
    
    # Visualize split distribution
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Split by velocity
    pd.crosstab(df['velocity_layer'], df['split']).plot(kind='bar', ax=ax1)
    ax1.set_title('Split distribution by velocity layer')
    ax1.set_xlabel('Velocity Layer')
    
    # Split by note range
    df['note_range'] = pd.cut(df['midi_note'], bins=[20, 35, 59, 83, 109], labels=['Bass', 'Low-Mid', 'Mid', 'High'])
    pd.crosstab(df['note_range'], df['split']).plot(kind='bar', ax=ax2)
    ax2.set_title('Split distribution by register')
    ax2.set_xlabel('Note Range')
    
    plt.tight_layout()
    plt.show()

## 5. Room IR Verification

In [ ]:
if FEATURES_H5_PATH.exists():
    with h5py.File(FEATURES_H5_PATH, 'r') as hf:
        # Find groups with room_ir
        ir_groups = [k for k in hf.keys() if 'room_ir' in hf[k]]
        print(f'Groups with room_ir: {len(ir_groups)}')
        
        if ir_groups:
            # Plot a sample IR
            sample_ir = hf[ir_groups[0]]['room_ir'][:]
            print(f'IR shape: {sample_ir.shape}')
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
            
            # Time domain
            t = np.arange(len(sample_ir)) / 44100
            ax1.plot(t * 1000, sample_ir)
            ax1.set_xlabel('Time (ms)')
            ax1.set_ylabel('Amplitude')
            ax1.set_title(f'Room IR: {ir_groups[0]}')
            ax1.grid(True, alpha=0.3)
            
            # Frequency response
            freqs = np.fft.rfftfreq(len(sample_ir), 1/44100)
            magnitude = np.abs(np.fft.rfft(sample_ir))
            ax2.semilogx(freqs[1:], 20 * np.log10(magnitude[1:] + 1e-10))
            ax2.set_xlabel('Frequency (Hz)')
            ax2.set_ylabel('Magnitude (dB)')
            ax2.set_title('Room IR Frequency Response')
            ax2.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()

## 6. Output Files Summary

In [ ]:
files = {
    'Features HDF5': FEATURES_H5_PATH,
    'Manifest Parquet': MANIFEST_PATH,
    'MIDI Pairs HDF5': MIDI_PAIRS_H5_PATH,
}

for name, path in files.items():
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f'{name}: EXISTS ({size_mb:.1f} MB)')
    else:
        print(f'{name}: MISSING')

# Run programmatic validation
from piano_neuronal.s1_validate.validation import validate_all
validate_all()